# 🗞️ Somali News Scraper
**Categories:** Siyaasad · Amni · Caalamka | **Target:** 10,000 articles per category

| Cell | Purpose |
|------|---------|
| 0    | Install dependencies |
| 1    | Imports & setup |
| 2    | Check current progress |
| 3    | Run full pipeline (all sites × all categories) |
| 4    | Scrape one specific site/category |
| 5    | Inspect raw data |
| 6    | Merge & export final datasets |
| 7    | Quality report |

## Cell 0 — Install Dependencies

In [1]:
# Run once. Restart kernel after installation.
import subprocess, sys

packages = [
    'requests',
    'beautifulsoup4',
    'lxml',
    'pandas',
    'tqdm',
]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = '✔' if result.returncode == 0 else '✘'
    print(f'{status} {pkg}')

print('\n✅ All packages installed. Restart the kernel now if this was your first run.')

✔ requests
✔ beautifulsoup4
✔ lxml
✔ pandas
✔ tqdm

✅ All packages installed. Restart the kernel now if this was your first run.


## Cell 1 — Imports & Setup

In [2]:
import scraper
import pipeline
import utils

print("scraper.py contains:")
print([x for x in dir(scraper) if not x.startswith("_")])

print("\npipeline.py contains:")
print([x for x in dir(pipeline) if not x.startswith("_")])

print("\nutils.py contains:")
print([x for x in dir(utils) if not x.startswith("_")])

scraper.py contains:
['BeautifulSoup', 'CFG', 'SCRAPER_CONFIG', 'USER_AGENTS', 'clean_text', 'extract_article', 'extract_article_links', 'fetch', 'get_logger', 'get_page_url', 'is_empty_listing', 'logger', 'make_session', 'polite_sleep', 'random', 're', 'requests', 'time', 'urljoin', 'urlparse']

pipeline.py contains:
['CATEGORIES', 'CFG', 'Checkpoint', 'DedupCache', 'RawWriter', 'SCRAPER_CONFIG', 'SITES', 'extract_article', 'extract_article_links', 'fetch', 'get_logger', 'get_page_url', 'get_status', 'is_empty_listing', 'logger', 'make_session', 'merge_raw_files', 'polite_sleep', 'print_progress', 'run_pipeline', 'scrape_site_category']

utils.py contains:
['BASE_DIR', 'CHECKPOINT_DIR', 'Checkpoint', 'DATA_DIR', 'DedupCache', 'FINAL_DIR', 'LOG_DIR', 'Path', 'RAW_DIR', 'RawWriter', 'csv', 'd', 'datetime', 'get_logger', 'hashlib', 'json', 'logging', 'merge_raw_files', 'os', 'polite_sleep', 'print_progress', 'random', 'time']


In [3]:
import sys
import os
import warnings
import importlib

warnings.filterwarnings("ignore")

# Make sure project folder is on the path
NOTEBOOK_DIR = os.path.abspath("")
if NOTEBOOK_DIR not in sys.path:
    sys.path.insert(0, NOTEBOOK_DIR)

import config
import utils
import scraper
import pipeline

# Reload fresh versions, useful after editing .py files
importlib.reload(config)
importlib.reload(utils)
importlib.reload(scraper)
importlib.reload(pipeline)

logger = utils.get_logger()

print("✅ Scraper loaded successfully.")
print(f"   Sites configured : {len(pipeline.SITES)}")
print(f"   Categories       : {pipeline.CATEGORIES}")

✅ Scraper loaded successfully.
   Sites configured : 15
   Categories       : ['siyaasad', 'amni', 'caalamka']


In [4]:
import pipeline
import inspect
import os

print("Current notebook folder:")
print(os.getcwd())

print("\nPipeline file being used:")
print(pipeline.__file__)

print("\nget_status() source currently loaded:")
print(inspect.getsource(pipeline.get_status))

Current notebook folder:
C:\Users\hp\Downloads\files

Pipeline file being used:
C:\Users\hp\Downloads\files\pipeline.py

get_status() source currently loaded:
def get_status() -> dict:
    """Return current article counts per (site, category). Call from notebook."""
    from utils import RAW_DIR
    import csv
    status = {}
    for fp in RAW_DIR.glob("*.csv"):
        stem   = fp.stem                       # e.g. "caasimada__siyaasad"
        parts  = stem.split("__", 1)
        if len(parts) != 2:
            continue
        site, cat = parts
        with open(fp, encoding="utf-8") as f:
            count = max(0, sum(1 for _ in f) - 1)
        status[f"{site}/{cat}"] = count
    return status



## Cell 2 — Check Current Progress

In [5]:
import pandas as pd
import pipeline
import utils

status = pipeline.get_status()

if not status:
    print("No data collected yet. Run Cell 3 or Cell 4 to start scraping.")
else:
    rows = []
    for key, count in sorted(status.items()):
        site, cat = key.split("/")
        rows.append({
            "Site": site,
            "Category": cat,
            "Articles": count
        })

    df = pd.DataFrame(rows)

    summary = df.groupby("Category")["Articles"].sum().reset_index()
    summary["Target"] = 10_000
    summary["Progress"] = (summary["Articles"] / summary["Target"] * 100).round(1).astype(str) + "%"
    summary["Remaining"] = (summary["Target"] - summary["Articles"]).clip(lower=0)

    print("\n📊 COLLECTION SUMMARY")
    print("=" * 55)
    print(summary.to_string(index=False))

    print("\n📋 PER-SITE BREAKDOWN")
    print("=" * 55)
    print(
        df.pivot_table(
            index="Site",
            columns="Category",
            values="Articles",
            aggfunc="sum",
            fill_value=0
        ).to_string()
    )

print(f"\n📁 Raw data folder  : {utils.RAW_DIR}")
print(f"📁 Checkpoints      : {utils.CHECKPOINT_DIR}")
print(f"📁 Final datasets   : {utils.FINAL_DIR}")


📊 COLLECTION SUMMARY
Category  Articles  Target Progress  Remaining
    amni      1548   10000    15.5%       8452
caalamka      5992   10000    59.9%       4008
siyaasad     10062   10000   100.6%          0

📋 PER-SITE BREAKDOWN
Category        amni  caalamka  siyaasad
Site                                    
caasimada          0      4049      5316
goobjoog        1548      1519      4746
mustaqbalmedia     0       424         0

📁 Raw data folder  : C:\Users\hp\Downloads\data\raw
📁 Checkpoints      : C:\Users\hp\Downloads\data\checkpoints
📁 Final datasets   : C:\Users\hp\Downloads\data\final


## Cell 3 — Run Full Pipeline
> Scrapes **all sites × all categories** automatically. Resumes from checkpoints if interrupted.

In [6]:
import pandas as pd

from pipeline import get_status
from utils import RAW_DIR, CHECKPOINT_DIR, FINAL_DIR

status = get_status()

if not status:
    print("No data collected yet. Run Cell 3 or Cell 4 to start scraping.")
else:
    rows = []
    for key, count in sorted(status.items()):
        site, cat = key.split("/")
        rows.append({
            "Site": site,
            "Category": cat,
            "Articles": count
        })

    df = pd.DataFrame(rows)

    # Summary by category
    summary = df.groupby("Category")["Articles"].sum().reset_index()
    summary["Target"] = 10_000
    summary["Progress"] = (summary["Articles"] / summary["Target"] * 100).round(1).astype(str) + "%"
    summary["Remaining"] = (summary["Target"] - summary["Articles"]).clip(lower=0)

    print("\n📊 COLLECTION SUMMARY")
    print("=" * 55)
    print(summary.to_string(index=False))

    print("\n📋 PER-SITE BREAKDOWN")
    print("=" * 55)
    print(
        df.pivot_table(
            index="Site",
            columns="Category",
            values="Articles",
            aggfunc="sum",
            fill_value=0
        ).to_string()
    )

print(f"\n📁 Raw data folder  : {RAW_DIR}")
print(f"📁 Checkpoints      : {CHECKPOINT_DIR}")
print(f"📁 Final datasets   : {FINAL_DIR}")


📊 COLLECTION SUMMARY
Category  Articles  Target Progress  Remaining
    amni      1548   10000    15.5%       8452
caalamka      5992   10000    59.9%       4008
siyaasad     10062   10000   100.6%          0

📋 PER-SITE BREAKDOWN
Category        amni  caalamka  siyaasad
Site                                    
caasimada          0      4049      5316
goobjoog        1548      1519      4746
mustaqbalmedia     0       424         0

📁 Raw data folder  : C:\Users\hp\Downloads\data\raw
📁 Checkpoints      : C:\Users\hp\Downloads\data\checkpoints
📁 Final datasets   : C:\Users\hp\Downloads\data\final


## Cell 4 — Scrape One Specific Site / Category
> Use this to target a specific site or top-up a category that's behind.

In [7]:
import requests
from bs4 import BeautifulSoup

import pipeline
import scraper

SITE = "goobjoog"
CATEGORY = "amni"

site_cfg = pipeline.SITES[SITE]
cat_url = site_cfg["categories"][CATEGORY]
page_url = scraper.get_page_url(cat_url, 1, site_cfg["pagination"])
link_sel = site_cfg["article_link_sel"]

print("SITE:", SITE)
print("CATEGORY:", CATEGORY)
print("Category URL:", cat_url)
print("Generated Page 1 URL:", page_url)
print("Article link selector:", link_sel)

session = scraper.make_session()

response = session.get(page_url, timeout=20, allow_redirects=True)

print("\nHTTP STATUS:", response.status_code)
print("Final URL:", response.url)
print("Content-Type:", response.headers.get("content-type"))
print("HTML length:", len(response.text))
print("\nFirst 500 characters:")
print(response.text[:500])

soup = BeautifulSoup(response.text, "html.parser")

print("\nPage title:")
print(soup.title.get_text(strip=True) if soup.title else "No title found")

matches = soup.select(link_sel)

print("\nSelector match count:", len(matches))

print("\nFirst matched links:")
for a in matches[:10]:
    print("-", a.get_text(" ", strip=True)[:100], "=>", a.get("href"))

print("\nFallback: all possible article-like links")
candidates = []

for a in soup.select("a[href]"):
    text = a.get_text(" ", strip=True)
    href = a.get("href")

    if text and href and len(text) > 20:
        candidates.append((text, href))

print("Candidate links found:", len(candidates))

for text, href in candidates[:30]:
    print("-", text[:100], "=>", href)

SITE: goobjoog
CATEGORY: amni
Category URL: https://goobjoog.com/qayb/amniga/
Generated Page 1 URL: https://goobjoog.com/qayb/amniga/
Article link selector: .jeg_post_title a, .jeg_post_title > a, h3.jeg_post_title a, h2.jeg_post_title a, .entry-title a, h2.entry-title a, h3.entry-title a

HTTP STATUS: 200
Final URL: https://goobjoog.com/qayb/amniga/
Content-Type: text/html; charset=UTF-8
HTML length: 131147

First 500 characters:
<!doctype html>
<html lang="so-SO">
<head>
	<meta charset="UTF-8">
	<meta name="viewport" content="width=device-width, initial-scale=1">
	<link rel="profile" href="https://gmpg.org/xfn/11">
	<style id="jetpack-boost-critical-css">@media all{@charset "UTF-8";ul{box-sizing:border-box}}@media all{@import "https://use.typekit.net/bpb5vat.css";*{box-sizing:border-box;border-width:0;border-style:solid;border-color:#e5e7eb}html{line-height:1.5;-webkit-text-size-adjust:100%;-moz-tab-size:4;-o-tab-size:4

Page title:
Arrimaha Amniga – Goobjoog

Selector match count: 0

In [8]:
from pathlib import Path
from utils import CHECKPOINT_DIR

dedup = CHECKPOINT_DIR / "dedup__amni.json"
print("Dedup cache exists:", dedup.exists())

Dedup cache exists: True


In [13]:
import importlib, scraper, pipeline
importlib.reload(scraper)
importlib.reload(pipeline)

# Quick sanity check — should print False (page is NOT empty)
import requests
from bs4 import BeautifulSoup
session = scraper.make_session()
r = session.get("https://goobjoog.com/qayb/amniga/", timeout=20)
soup = BeautifulSoup(r.text, "html.parser")
result = scraper.is_empty_listing(soup, ".jeg_post_title a", "https://goobjoog.com")
print("is_empty_listing returned:", result)   # must print False

is_empty_listing returned: False


In [14]:
import importlib
import config, scraper, pipeline

importlib.reload(config)
importlib.reload(scraper)
importlib.reload(pipeline)

# Now scrape
SITE     = "goobjoog"
CATEGORY = "amni"
TARGET   = 10000
MAX_PAGES = 1300

print(f"Scraping  {SITE} / {CATEGORY}  (target: {TARGET:,} articles)")

n = pipeline.scrape_site_category(
    site_name=SITE,
    category=CATEGORY,
    target=TARGET,
    max_pages=MAX_PAGES,
    force_restart=False,
)
print(f"\n✔ Done. {n:,} new articles written in this run.")

2026-05-18 10:01:19 | INFO     | ▶ [goobjoog] [amni] Starting at page 1 | checkpoint: 1548 | writer: 1548


Scraping  goobjoog / amni  (target: 10,000 articles)


2026-05-18 10:01:21 | INFO     |   [goobjoog][amni] Page 1: 28 links found
2026-05-18 10:02:12 | WARNING  | Connection error (HTTPSConnectionPool(host='goobjoog.comm', port=443): Max retries exceeded with url: /english (Caused by NameResolutionError("HTTPSConnection(host='goobjoog.comm', port=443): Failed to resolve 'goobjoog.comm' ([Errno 11001] getaddrinfo failed)"))). Waiting 2.0s… (attempt 1/4)
2026-05-18 10:02:14 | WARNING  | Connection error (HTTPSConnectionPool(host='goobjoog.comm', port=443): Max retries exceeded with url: /english (Caused by NameResolutionError("HTTPSConnection(host='goobjoog.comm', port=443): Failed to resolve 'goobjoog.comm' ([Errno 11001] getaddrinfo failed)"))). Waiting 4.0s… (attempt 2/4)
2026-05-18 10:02:18 | WARNING  | Connection error (HTTPSConnectionPool(host='goobjoog.comm', port=443): Max retries exceeded with url: /english (Caused by NameResolutionError("HTTPSConnection(host='goobjoog.comm', port=443): Failed to resolve 'goobjoog.comm' ([Errno 1100


✔ Done. 5,047 new articles written in this run.


## Cell 5 — Inspect Raw Data

In [10]:
import pandas as pd
from utils import RAW_DIR

# Load a single raw file to inspect
INSPECT_SITE     = 'caasimada'
INSPECT_CATEGORY = 'siyaasad'

fp = RAW_DIR / f'{INSPECT_SITE}__{INSPECT_CATEGORY}.csv'

if not fp.exists():
    print(f'File not found: {fp}')
    print('Run Cell 3 or Cell 4 first.')
else:
    df = pd.read_csv(fp)
    print(f'📄 {fp.name}')
    print(f'   Rows    : {len(df):,}')
    print(f'   Columns : {list(df.columns)}')
    print(f'   Date range: {df["scraped_at"].min()} → {df["scraped_at"].max()}\n')
    
    print('── Sample titles (first 10) ──────────────────────────────')
    for i, row in df.head(10).iterrows():
        print(f'  [{i+1:02d}] {row["title"]}')

    print(f'\n── Word count distribution ───────────────────────────────')
    print(df['word_count'].describe().round(1).to_string())

📄 caasimada__siyaasad.csv
   Rows    : 5,316
   Columns : ['id', 'site', 'category', 'url', 'title', 'content', 'scraped_at', 'word_count']
   Date range: 2026-05-16 09:10:46 → 2026-05-17 11:02:24

── Sample titles (first 10) ──────────────────────────────
  [01] DF iyo M/Yurub billaabaya wada-hadallo ku saabsan soo-celinta Soomaalida
  [02] Burcad-badeeda oo faa’iido dhaqaale ka helay dagaalka Iran
  [03] Shacabka Muqdisho oo dhibaato ku qaba helitaanka kaarka aqoonsiga ee NIRA
  [04] Sawirro: Madaxweyne Xasan Sheekh oo xarigga ka jaray…
  [05] Daawo Shariif: “Xasan Sheekh wuxuu taagan yahay meeshii uu ku dhiman lahaa”
  [06] Golaha Mustaqbalka oo Xasan Sheekh u aqoonsaday madaxweyne hore
  [07] Wada-hadalladii Xalane oo fashilmay iyo DF oo ku dhawaaqday in dalku galay…
  [08] Xog: Lacago laaluush ah oo garoonka Muqdisho looga qaado dadka u dhoofa…
  [09] Golaha Mustaqbalka oo war cusub kaso saaray qabsoomida banaanbaxa Muqdisho
  [10] DF oo shaacisay inay duqeysay ciidankii Lafta-gar

## Cell 6 — Merge & Export Final Datasets
> Combines all per-site CSVs into one clean file per category, with cross-site deduplication.

In [11]:
import pandas as pd
from scraper.utils import merge_raw_files, FINAL_DIR

CATEGORIES_TO_MERGE = ['siyaasad', 'amni', 'caalamka']

for cat in CATEGORIES_TO_MERGE:
    out = merge_raw_files(cat)
    if out and out.exists():
        df = pd.read_csv(out)
        print(f'  ✔ {cat:<12} → {len(df):>6,} unique articles saved to {out.name}')

print(f'\n📁 Final files in: {FINAL_DIR}')

ModuleNotFoundError: No module named 'scraper.utils'; 'scraper' is not a package

## Cell 7 — Quality Report

In [ ]:
import pandas as pd
from pathlib import Path
from scraper.utils import FINAL_DIR

print('=' * 65)
print('  DATASET QUALITY REPORT')
print('=' * 65)

all_dfs = []
for fp in sorted(FINAL_DIR.glob('*_final.csv')):
    df = pd.read_csv(fp)
    all_dfs.append(df)
    cat = fp.stem.replace('_final', '')

    missing_titles   = df['title'].isna().sum()
    missing_content  = df['content'].isna().sum()
    short_titles     = (df['title'].str.len() < 20).sum()
    short_content    = (df['word_count'] < 50).sum()
    avg_words        = df['word_count'].mean()
    sites_covered    = df['site'].nunique()
    
    print(f'\n  Category       : {cat.upper()}')
    print(f'  Total articles : {len(df):,}')
    print(f'  Sites          : {sites_covered} ({df["site"].value_counts().to_dict()})')
    print(f'  Avg word count : {avg_words:.0f} words')
    print(f'  Missing titles : {missing_titles}')
    print(f'  Missing content: {missing_content}')
    print(f'  Short titles   : {short_titles} (< 20 chars)')
    print(f'  Short content  : {short_content} (< 50 words)')

if all_dfs:
    combined = pd.concat(all_dfs)
    print(f'\n  TOTAL across all categories: {len(combined):,} articles')
    print(f'  Category balance:')
    print(combined['category'].value_counts().to_string())

print('\n' + '=' * 65)